In [ ]:
import stim
import logging

from IPython.core.display import Markdown

logging.basicConfig(level=logging.ERROR)
from library.circuitry import Circuitry
from library.common import Pauli
from library.qubit_array import QubitArray
from library.steane_code.patch import SteaneCodePatch
from utils.simulation.stim import simulate, sample

In [ ]:
def detector_report(circuitry: stim.Circuit) -> str:
    issues = []
    if len(circuitry.missing_detectors()) > 0:
        issues.append("MISSING")

    try:
        circuitry.detector_error_model(allow_gauge_detectors=False)
    except ValueError:
        issues.append("GAUGE")

    return "&".join(issues)

In [ ]:
scenarios : dict[str, stim.Circuit] = {}

In [ ]:
def append_cultivation(steane: SteaneCodePatch, circuitry: Circuitry):
    all_qubits = steane.qubits
    meas_qubits = [ steane.qubits[q] for q in [ 10, 13, 14, 15] ]
    circuitry.append(steane.injection.name + "_DAG", all_qubits[1])
    circuitry.append("H", [all_qubits[0], all_qubits[3]])
    circuitry.append("RX", meas_qubits)
    circuitry.append_tick()

    # Do the thing.
    circuitry.append("CX", [ all_qubits[q] for q in [ 10, 5, 13, 1, 15, 3 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 6, 13, 0, 15 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 6, 10, 14, 0 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 14, 6 ]])
    circuitry.append_tick()
    circuitry.append("MX", meas_qubits[2])
    qubits.record_measurement(meas_qubits[2], f"CULT:X0")
    circuitry.append_tick()
    # Do the thing in reverse.
    circuitry.append("RX", meas_qubits[2])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 14, 6 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 6, 10, 14, 0 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 6, 13, 0, 15 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 10, 5, 13, 1, 15, 3 ]])
    circuitry.append_tick()

    circuitry.append("MX", meas_qubits)
    for idx, qa in enumerate(meas_qubits):
        qubits.record_measurement(qa, f"CULT:X{idx+1}")

    circuitry.append("H", [all_qubits[0], all_qubits[3]])
    circuitry.append(steane.injection.name, all_qubits[1])
    circuitry.append_tick()

In [ ]:
qubits = QubitArray(dimensions=(5,3))
circuitry = Circuitry(qubits)
steane = SteaneCodePatch(qubits)
steane.append_preparation(circuitry)
for rnd in range(3):
    steane.append_superdense_cycle(circuitry, prefix=f"SDC{rnd}")
append_cultivation(steane, circuitry)

steane.annotate_detectors(circuitry, sdc_rounds=3)

circuitry.append_observable(0, "OBSERVABLE_CULTIVATION", {
    steane.qubits[5] : "X", steane.qubits[6] : "X", steane.qubits[1] : "Y", steane.qubits[0] : "Z" , steane.qubits[3] : "Z"
})

scenarios['Weight-5'] = circuitry.as_stim

for detector in circuitry.missing_detectors():
    records = list(map(lambda neg: qubits.retrieve_record(neg.value), detector.targets_copy()))
    print(f">> Detector : {records}")

In [ ]:
qubits = QubitArray(dimensions=(5,3))
circuitry = Circuitry(qubits)
steane = SteaneCodePatch(qubits)
steane.append_preparation(circuitry)
for rnd in range(3):
    steane.append_superdense_cycle(circuitry, prefix=f"SDC{rnd}")
steane.append_cultivation(circuitry)
steane.annotate_detectors(circuitry, sdc_rounds=3)

circuitry.append_observable(0, "OBSERVABLE_CULTIVATION", steane.logical(Pauli.Y))

scenarios["Weight-7"] = circuitry.as_stim

for detector in circuitry.missing_detectors():
    records = list(map(lambda neg: qubits.retrieve_record(neg.value), detector.targets_copy()))
    print(f">> Detector : {records}")

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    warning = detector_report(circuitry)
    display(Markdown(f"[Open in Crumble (Point {index} - {scenario})]({circuitry.to_crumble_url()}) {warning}"))

In [ ]:
sample(scenarios, title=r"Cultivation stage for $|\mathbf{S}\rangle$ [$\overline{\mathbf{Y}}$-observable]", label="Circuitry")

In [ ]:
simulate(scenarios, postselection=True, title=r"Cultivation stage for $|\mathbf{S}\rangle$", label="Circuitry", shots=1e7, num_workers=7)